# Imports

In [26]:
# -*- coding: utf-8 -*-
"""
Created on Wed Oct 11 16:05:15 2023

@author: andrej
"""
import sys,os
sys.path.append(r'C:/data/EnergyTrading/Python/')

import pandas as pd
from Database.TPData import TPData, TPDataDa
from Database.DB_reader import Database
from OrderBook.OrderBook import OrderBookSnaps
from datetime import date, timedelta, datetime

import cx_Oracle
try:
    cx_Oracle.init_oracle_client(lib_dir=r"C:\Users\user\Downloads\instantclient_21_11")
except:
    pass

l_path = '//192.168.10.91/data/Data/orderbooks/base/'
local_db_config_path=r'C:\data\EnergyTrading\configDB.json'

In [12]:
def load_ob(m, t, dt, p_d, bT, eT):
    ob_class = OrderBookSnaps(verbose=True)
    file_path = l_path + t.split('_')[0] + '/'
    file_name = m + '_' + t.split('_')[0] + '_' + p_d.strftime('%y%m%d') + '_' + dt.strftime('%y%m%d') + '.p'
    print(('%s Loading OrderBook %d...' % (dt.strftime('%y-%m-%d'), 0)))
    time_load = ob_class.import_data(file_path + file_name)
    print(('OrderBook %d created in %d sec' % (0, time_load)))
    # ob_class.LoB_truncate(thres_vol=1)
    return ob_class.LoB_select(bT, eT, freq=None)

# Email utilities

In [13]:
from Utilities.email_sending import send_plain_email, send_html_email

EMAIL_PASSWORD = os.getenv('EMAIL_PASSWORD') # the password needs to be set as EMAIL_PASSWORD in system variables of the computer where the process is running, it is located in S:/Algo/email_password.txt
if EMAIL_PASSWORD is None:
    raise ValueError("EMAIL_PASSWORD environment variable not set")

RECIPIENT = "zubal_andrej@energytrading.sk" # can also be a list of recipients

# Calculate Bid - Ask best prices

In [57]:
mkt = 'de'
tenor = 'y'
prod = 'base'
venue_list = ['eex']
start_date1 = datetime(2025, 1, 1)
start_date2 = None
bT = datetime(2024, 6, 20, hour=8, minute=0, second=0)
eT = datetime(2024, 6, 20, hour=18, minute=0, second=0)

In [58]:
ob_class = OrderBookSnaps(verbose=True)
data_class = TPDataDa()
LoB_dict, time_list = load_ob(mkt, tenor, bT, start_date1, bT, eT)
ob_class.update_data(LoB_dict, time_list)

24-06-20 Loading OrderBook 0...
OrderBook 0 created in 18 sec


In [59]:
xx = pd.concat([ob_class.best_bid_all, ob_class.best_ask_all], axis=1)
xx.columns = ['bidbestprice', 'askbestprice']
xx1 = data_class.process_best_orders(xx)
 

In [60]:
datetime.now()

datetime.datetime(2024, 7, 2, 13, 36, 28, 304891)

In [61]:
df=xx1.reset_index(names='datetime')
df['instkey']='10641710_10000106'
df['firstsequenceitemid']=26
df['secondsequenceitemid']=None
df['upload_timestamp']=datetime.now()
df=df[['datetime', 'instkey', 'firstsequenceitemid', 'secondsequenceitemid', 'bidbestprice', 'askbestprice', 'upload_timestamp']]

In [62]:
df.head()

,datetime,instkey,firstsequenceitemid,secondsequenceitemid,bidbestprice,askbestprice,upload_timestamp
0,2024-06-20 08:00:00.558,10641710_10000106,26,None,89.50,96.25,2024-07-02 13:36:29.214809
1,2024-06-20 08:01:59.749,10641710_10000106,26,None,93.50,96.25,2024-07-02 13:36:29.214809
2,2024-06-20 08:02:24.454,10641710_10000106,26,None,93.50,95.50,2024-07-02 13:36:29.214809
3,2024-06-20 08:04:31.296,10641710_10000106,26,None,93.51,95.50,2024-07-02 13:36:29.214809
4,2024-06-20 08:04:40.649,10641710_10000106,26,None,93.52,94.50,2024-07-02 13:36:29.214809


In [63]:
conn = Database()
conn._connect()
df.to_sql('ba_price', conn.engine, schema='best_orders', if_exists='append', index=False)

Connected to the database postgre


661

In [ ]:
# local_db_config_path=r'C:\data\EnergyTrading\configDB.json'

try:
    for date in date_range:
        
        conn = Database('OracleSQL',path_name=local_db_config_path)

        query=f"""select * from  rove_od.trayport_vw_trades 
        WHERE 1=1
        and to_date(datetime)>=to_date('{date}', 'YYYY-MM-DD')
        and to_date(datetime)<=to_date('{date}', 'YYYY-MM-DD')""" # where rownum <= 100"""
        #query="""select * from  public.trayport_orders limit 100"""

        df1=conn.execute(query)

        conn = Database(path_name=local_db_config_path)
        conn._connect()
        df1.to_sql('trayport_vw_trades', conn.engine, schema='public', if_exists='append', index=False)

        print('\n')
        print(f'{date} number of records: {df1.shape[0]}')

        #email sending in case of success, also sending number of records uploaded
        send_plain_email(
          RECIPIENT, 
        "SUCCESS: trade_data_daily_update_remote_comp job", f'{date} number of records: {df1.shape[0]}',
        email_password=EMAIL_PASSWORD
    )
        print('\n')

        conn._disconnect()

except:
    send_plain_email(
          RECIPIENT, 
        "FAIL: trade_data_daily_update_remote_comp", f'The upload of data failed for this run, please check what is the issue.',
        email_password=EMAIL_PASSWORD
    )